<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 01 — Tokenisation, Text Normalisation & Datasets

**Course:** Natural Language Processing  
**Session:** 1 / 10  
**Duration:** ~2h (TP portion)

---

## Objectives

By the end of this session you will be able to:

- Apply a reproducible text normalisation pipeline
- Compare tokenisation strategies and measure their impact on vocabulary size and OOV rate
- Load a text classification dataset, inspect it critically, and build a proper train/val/test split
- Implement a majority-class baseline and interpret the resulting metrics
- Write a short error analysis

---

## Roadmap

| Part | Task | Deliverable |
|------|------|-------------|
| 1 | Text normalisation | Implement `preprocess_text` + pass assertions |
| 2 | Tokenisation comparison | Stats table: vocab size, mean length, OOV rate |
| 3 | Dataset inspection & split | Class distribution plot + justified split |
| 4 | Majority-class baseline | Classification report + confusion matrix |
| 5 | Error analysis | Written analysis of 10 misclassified examples |

Each part ends with a **📝 Your analysis** cell. Fill these in — they are reviewed at the end of the session.

---

## Ground rules

- Every function must have its docstring filled in **before** you write the body.
- Do not look at the test set before Part 4.
- When you hit an error, read it carefully before asking for help.

---

## Setup

In [ ]:
# Uncomment if running on Colab
# !pip install datasets -q

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Dataset ───────────────────────────────────────────────────────────────────
# Your instructor will announce the dataset at the start of class.
# Supported values:
#   "ag_news"   → 4-class news topic classification (recommended for this session)
#   "imdb"      → binary sentiment
#   "emotion"   → 6-class emotion in tweets  (dair-ai/emotion)
#
DATASET_NAME = "ag_news"   # ← CHANGE THIS IF INSTRUCTED

print(f"Dataset  : {DATASET_NAME}")
print("Setup OK.")

---
## Part 1 — Text Normalisation

Before any model sees text, it goes through a normalisation pipeline. Small decisions here (lowercase or not? keep punctuation?) have downstream effects on vocabulary size and model performance.

We will build **one reusable function** and use it throughout the course.

### 1.1 — Implement `preprocess_text`

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Apply a minimal, reproducible normalisation pipeline to a single string.

    Steps (apply in this exact order):
      1. Lowercase all characters
      2. Remove HTML tags (e.g. <br />, <p>, </div>)
      3. Replace any character that is not a letter, digit, or space
         with a single space
      4. Collapse multiple consecutive whitespace characters into one space
      5. Strip leading and trailing whitespace

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string.

    Examples
    --------
    >>> preprocess_text("Hello,   World! <br/>")
    'hello world'
    >>> preprocess_text("  NLP is FUN!!!  ")
    'nlp is fun'
    >>> preprocess_text("<p>U.S. stocks rose 2.3%</p>")
    'u s stocks rose 2 3'
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# ── Assertions — all must pass before you continue ────────────────────────────
assert preprocess_text("Hello,   World! <br/>")       == "hello world",       "Failed: HTML + punctuation"
assert preprocess_text("  NLP is FUN!!!  ")            == "nlp is fun",        "Failed: leading/trailing spaces + punctuation"
assert preprocess_text("<p>U.S. stocks rose 2.3%</p>") == "u s stocks rose 2 3", "Failed: HTML + mixed punctuation"
assert preprocess_text("already clean")               == "already clean",     "Failed: clean input should be unchanged"
assert preprocess_text("")                             == "",                  "Failed: empty string"
print("All assertions passed ✓")

### 1.2 — Ablation: what happens if we skip a step?

Understanding *why* each step matters is as important as implementing it.

In [ ]:
def preprocess_no_lowercase(text: str) -> str:
    """
    Same pipeline as preprocess_text but WITHOUT the lowercase step.

    Used to measure the effect of case folding on vocabulary size.
    """
    # YOUR CODE HERE  (copy-paste and remove step 1)
    raise NotImplementedError


def preprocess_keep_punctuation(text: str) -> str:
    """
    Same pipeline as preprocess_text but keeping punctuation characters.

    Only apply steps 1, 2, 4, and 5 (skip the punctuation removal step).
    """
    # YOUR CODE HERE
    raise NotImplementedError

### 📝 Your analysis — Part 1

**Which step in the pipeline has the biggest impact on vocabulary size, and why?**

> *[Your answer here]*

**Give an example where removing punctuation loses information** that a classifier might need.

> *[Your answer here]*

---
## Part 2 — Tokenisation Comparison

Tokenisation is the first design decision in any NLP pipeline. We will compare three strategies on the same corpus and measure their properties quantitatively.

The three tokenisers:

| Name | Strategy | Library |
|------|----------|---------|
| `whitespace` | Split on spaces | pure Python |
| `nltk` | Rule-based word tokeniser | `nltk` |
| `bpe` | GPT-4 byte-pair encoding | `tiktoken` |

In [ ]:
# Uncomment to install if needed
# !pip install nltk tiktoken -q

import nltk
import tiktoken
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Initialise the BPE tokeniser once (it's slow to load)
BPE_ENC = tiktoken.encoding_for_model("gpt-4")

### 2.1 — Implement tokeniser wrappers

In [ ]:
def tokenize_whitespace(text: str) -> list[str]:
    """
    Tokenise by splitting on whitespace.

    Apply preprocess_text first, then split on spaces.
    Returns a list of string tokens.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def tokenize_nltk(text: str) -> list[str]:
    """
    Tokenise using NLTK's word_tokenize.

    Apply preprocess_text first, then use nltk.word_tokenize.
    Returns a list of string tokens.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def tokenize_bpe(text: str) -> list[int]:
    """
    Tokenise using GPT-4's BPE encoding (tiktoken).

    Do NOT apply preprocess_text first — BPE operates on raw text.
    Returns a list of integer token IDs.

    Hint: use BPE_ENC.encode(text)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Quick sanity check
sample = "The Federal Reserve raised interest rates by 0.25%."
print("whitespace :", tokenize_whitespace(sample))
print("nltk       :", tokenize_nltk(sample))
print("bpe        :", tokenize_bpe(sample))

### 2.2 — Build the comparison table

We compare the tokenisers on a sample of the corpus. Computing over the full dataset would take too long for BPE.

In [ ]:
def compare_tokenisers(
    texts: list[str],
    tokenisers: dict,
) -> pd.DataFrame:
    """
    Compute statistics for each tokeniser on a list of texts.

    For each tokeniser, compute:
      - vocab_size    : number of unique tokens across all texts
      - mean_length   : mean number of tokens per text
      - median_length : median number of tokens per text
      - oov_proxy     : fraction of unique tokens that appear only once
                        (hapax legomena — a proxy for OOV rate on unseen data)

    Parameters
    ----------
    texts      : list of raw strings
    tokenisers : dict mapping name (str) → callable that takes str and returns list

    Returns
    -------
    pd.DataFrame with one row per tokeniser and the four columns above.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Load a small sample to keep runtime under 60 seconds
# (we'll load the full dataset in Part 3)
sample_raw = load_dataset(DATASET_NAME, split="train[:500]")
sample_texts = sample_raw["text"]

tokenisers = {
    "whitespace": tokenize_whitespace,
    "nltk"      : tokenize_nltk,
    "bpe"       : tokenize_bpe,
}

comparison = compare_tokenisers(sample_texts, tokenisers)
comparison

### 2.3 — Vocabulary growth curve

How quickly does each tokeniser's vocabulary saturate as we see more text?

In [ ]:
def plot_vocab_growth(
    texts: list[str],
    tokenisers: dict,
    step: int = 20,
) -> None:
    """
    Plot vocabulary size as a function of number of texts seen.

    For each tokeniser, iterate over texts in chunks of `step`,
    and after each chunk record the cumulative vocabulary size.
    Plot one line per tokeniser.

    Parameters
    ----------
    texts      : list of raw strings
    tokenisers : dict mapping name → callable
    step       : record vocabulary size every `step` texts
    """
    # YOUR CODE HERE
    raise NotImplementedError


plot_vocab_growth(sample_texts, tokenisers, step=25)

### 📝 Your analysis — Part 2

**Compare the three tokenisers on vocabulary size and mean length.** Which produces the largest vocabulary? Which produces the shortest sequences? Is there a trade-off?

> *[Your answer here]*

**Why does BPE operate on raw text rather than preprocessed text?** What would happen if you applied `preprocess_text` before BPE?

> *[Your answer here]*

**Which tokeniser would you choose** for (a) a classical ML model like logistic regression, and (b) a Transformer? Justify briefly.

> *[Your answer here]*

---
## Part 3 — Dataset Inspection & Split

A model is only as good as the data it is trained on. Before fitting anything, we need to understand what we are working with.

### 3.1 — Load the full dataset

In [ ]:
def load_text_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    """
    Load a HuggingFace text classification dataset into a unified DataFrame.

    Returns
    -------
    df : pd.DataFrame
        Columns: 'text' (str), 'label' (int), 'label_name' (str), 'split' (str)
    meta : dict
        Keys: 'label_names' (list[str]), 'num_classes' (int), 'total_examples' (int)
    """
    configs = {
        "ag_news" : {"label_names": ["World", "Sports", "Business", "Sci/Tech"], "text_col": "text"},
        "imdb"    : {"label_names": ["Negative", "Positive"],                   "text_col": "text"},
        "emotion" : {"label_names": ["sadness", "joy", "love", "anger", "fear", "surprise"], "text_col": "text"},
    }
    if name not in configs:
        raise ValueError(f"Unknown dataset '{name}'. Choose from: {list(configs)}")

    cfg = configs[name]
    raw = load_dataset(name if name != "emotion" else "dair-ai/emotion")

    frames = []
    for split_name, split_data in raw.items():
        df_split = split_data.to_pandas().rename(columns={cfg["text_col"]: "text"})
        df_split["split"]      = split_name
        df_split["label_name"] = df_split["label"].apply(lambda i: cfg["label_names"][i])
        frames.append(df_split[["text", "label", "label_name", "split"]])

    df   = pd.concat(frames, ignore_index=True)
    meta = {"label_names": cfg["label_names"], "num_classes": len(cfg["label_names"]), "total_examples": len(df)}
    return df, meta


df, meta = load_text_dataset(DATASET_NAME)
print(f"Total   : {meta['total_examples']:,}")
print(f"Classes : {meta['label_names']}")
print(f"Splits  : {df['split'].value_counts().to_dict()}")
df.head(3)

### 3.2 — Compute per-class statistics

In [ ]:
def compute_dataset_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-class statistics on the training split.

    For each class compute:
      count, proportion (0–1), mean_length, median_length, min_length, max_length
    where 'length' = number of whitespace-split tokens in the raw text.

    Parameters
    ----------
    df : full dataset DataFrame (all splits)

    Returns
    -------
    pd.DataFrame — one row per class
    """
    # YOUR CODE HERE
    raise NotImplementedError


stats = compute_dataset_stats(df)
stats

### 3.3 — Visualise class distribution and text lengths

In [ ]:
def plot_class_distribution(df: pd.DataFrame, split: str = "train") -> None:
    """
    Bar chart of class counts for a given split.

    Requirements:
      - Bars sorted by frequency (descending)
      - Count label on top of each bar
      - Title includes split name and total count
    """
    # YOUR CODE HERE
    raise NotImplementedError


def plot_length_distribution(df: pd.DataFrame, split: str = "train", max_tokens: int = 200) -> None:
    """
    Overlapping histograms of whitespace-token counts, one per class.

    Requirements:
      - Clip lengths at max_tokens for readability
      - Semi-transparent bars (alpha=0.5)
      - Legend, axis labels, title
    """
    # YOUR CODE HERE
    raise NotImplementedError


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plot_class_distribution(df, split="train")
plt.sca(axes[1]); plot_length_distribution(df, split="train")
plt.tight_layout()

### 3.4 — Train / validation / test split

In [ ]:
def make_splits(
    df: pd.DataFrame,
    val_size: float = 0.1,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build train / validation / test DataFrames.

    Rules:
      - Reuse existing 'test' split if present; do not touch it.
      - Reuse existing 'validation' split if present.
      - If no 'validation' exists, carve one out of 'train'
        using stratified sampling (val_size fraction).

    Returns
    -------
    train_df, val_df, test_df
    """
    # YOUR CODE HERE
    raise NotImplementedError


train_df, val_df, test_df = make_splits(df, val_size=0.1)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")

# Verify stratification
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    props = split["label_name"].value_counts(normalize=True).round(3).to_dict()
    print(f"{name}: {props}")

### 📝 Your analysis — Part 3

**Describe the dataset in 3 sentences.** Size, task, class balance, typical text length.

> *[Your answer here]*

**Flag one potential issue** in the data that could affect training or evaluation.

> *[Your answer here]*

**Why must the test split be stratified?** What would happen to the reported accuracy if one class were over-represented in the test set?

> *[Your answer here]*

---
## Part 4 — Majority-Class Baseline

> *"Always compare against the stupidest possible model first."*

A majority-class baseline predicts the most frequent training label for every input, ignoring the text entirely. Any model that cannot beat this baseline does not have a learning problem — it has a data problem.

### 4.1 — Implement and evaluate

In [ ]:
def majority_baseline(
    train_df: pd.DataFrame,
    eval_df: pd.DataFrame,
) -> tuple[list, list]:
    """
    Predict the most frequent class in train_df for every example in eval_df.

    Returns
    -------
    y_true : list[int] — ground truth labels from eval_df
    y_pred : list[int] — predicted labels (constant, equal to majority class)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Evaluate on validation set first
y_true_val, y_pred_val = majority_baseline(train_df, val_df)
print("=== Validation ===")
print(classification_report(y_true_val, y_pred_val, target_names=meta["label_names"], zero_division=0))

In [ ]:
# ── Test set — run once, do not use results to tune anything ──────────────────
y_true_test, y_pred_test = majority_baseline(train_df, test_df)
print("=== Test ===")
print(classification_report(y_true_test, y_pred_test, target_names=meta["label_names"], zero_division=0))

### 4.2 — Confusion matrix

In [ ]:
def plot_confusion_matrix(
    y_true: list,
    y_pred: list,
    label_names: list[str],
    title: str = "Confusion Matrix",
    normalise: bool = True,
) -> None:
    """
    Plot a confusion matrix.

    When normalise=True, rows are normalised to sum to 1
    (i.e. each cell shows recall for that class).
    Display values with 2 decimal places when normalised, integers otherwise.
    Use a diverging colourmap (e.g. 'Blues').
    """
    # YOUR CODE HERE
    raise NotImplementedError


plot_confusion_matrix(y_true_test, y_pred_test, meta["label_names"], title="Majority Baseline — Test")

### 📝 Your analysis — Part 4

**Interpret macro-F1 vs weighted-F1.** They are different numbers here. Why? Which should you report?

> *[Your answer here]*

**Theoretical ceiling.** On a perfectly balanced K-class dataset, what is the maximum accuracy a majority classifier can achieve? How does your dataset compare to this ceiling?

> *[Your answer here]*

---
## Part 5 — Error Analysis

Error analysis is how you turn metric numbers into insight. Even on a trivial baseline, it tells you which examples the model systematically ignores — and why.

### 5.1 — Sample misclassified examples

In [ ]:
def find_errors(
    eval_df: pd.DataFrame,
    y_true: list,
    y_pred: list,
    label_names: list[str],
    n: int = 10,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Return a stratified sample of misclassified examples.

    'Stratified' means the n examples cover multiple true classes —
    not all from the same class.

    Returns a DataFrame with columns:
      'text'       : raw text
      'true_label' : ground truth class name
      'predicted'  : predicted class name
      'num_tokens' : whitespace token count
    """
    # YOUR CODE HERE
    raise NotImplementedError


errors = find_errors(test_df, y_true_test, y_pred_test, meta["label_names"], n=10)
pd.set_option("display.max_colwidth", 120)
errors

### 5.2 — Vocabulary overlap between classes

In [ ]:
def compute_vocab_overlap(train_df: pd.DataFrame, top_n: int = 500) -> pd.DataFrame:
    """
    Pairwise Jaccard similarity of per-class vocabularies.

    For each class, build a vocabulary from its top_n most frequent
    whitespace-split tokens. Then compute:

        J(A, B) = |vocab_A ∩ vocab_B| / |vocab_A ∪ vocab_B|

    Returns a symmetric DataFrame (rows = cols = class names).
    """
    # YOUR CODE HERE
    raise NotImplementedError


overlap = compute_vocab_overlap(train_df, top_n=500)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(overlap, annot=True, fmt=".2f", cmap="Blues", ax=ax, vmin=0, vmax=1)
ax.set_title("Vocabulary Jaccard Overlap — top-500 tokens per class")
plt.tight_layout()
plt.show()

### 📝 Your analysis — Part 5

**Describe a pattern in the 10 errors.** Do they share surface characteristics (length, topic, vocabulary)? Why does the majority baseline fail on them specifically?

> *[Your answer here]*

**What does the vocabulary overlap tell you?** Which pair of classes overlaps most? Does this match your intuition about the task?

> *[Your answer here]*

**One thing that surprised you** about this dataset that you did not expect at the start.

> *[Your answer here]*

---
## Summary

Run this cell to auto-generate your results table before the end-of-session review.

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Weighted F1"],
    "Majority Baseline (test)": [
        f"{accuracy_score(y_true_test, y_pred_test):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='macro',    zero_division=0):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0):.3f}",
    ],
})

print(f"Dataset    : {DATASET_NAME}")
print(f"Train size : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}")
print(f"Classes    : {meta['num_classes']}  ({', '.join(meta['label_names'])})")
print()
display(summary)
print()
print("Tokeniser comparison:")
display(comparison)

---
## Part 6 — Data Analysis Exercises

These exercises use the preprocessed text and splits you built in Parts 1–4.
They are designed in order of complexity.

| Exercise | Concepts | When |
|----------|----------|------|
| 6.1 Vocabulary & frequency | `Counter`, Zipf's law, bar charts | In class |
| 6.2 Tokeniser audit | OOV rate, sequence length distributions | In class |
| 6.3 Preprocessing ablation | Vocabulary size, χ² of top tokens | In class |
| 6.4 Class fingerprinting | Per-class TF-IDF, discriminative tokens | **Homework** |

All four exercises produce output that belongs in your **comparison table** — you will refer back to these numbers in every future session.

---
### Exercise 6.1 — Vocabulary & frequency analysis

Before fitting any model, understand your data by counting things.
You will:
1. Build a token frequency distribution for the **full training set** and per class.
2. Plot the **top-20 tokens per class** as horizontal bar charts.
3. Plot the **Zipf curve** for the full training vocabulary.
4. Measure **vocabulary overlap** between classes.

> **Expected insight:** a few dozen tokens dominate token counts (Zipf), and the most
> frequent tokens are stop words — class-neutral noise. The interesting tokens sit
> further down the frequency list.

In [ ]:
def build_vocab_freq(
    df: pd.DataFrame,
    text_col: str = "text",
    label_col: str = "label",
    label_names: list[str] | None = None,
) -> dict[str, Counter]:
    """Build per-class and global token frequency counters.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with at least ``text_col`` (preprocessed) and ``label_col`` columns.
    text_col : str
        Column containing preprocessed, whitespace-tokenised text.
    label_col : str
        Column containing integer class labels.
    label_names : list[str] or None
        Human-readable label names. If None, uses str(label).

    Returns
    -------
    dict[str, Counter]
        Keys: each label name + ``"__all__"`` for the global counter.
        Values: ``Counter`` mapping token → count.
    """
    if label_names is None:
        label_names = [str(l) for l in sorted(df[label_col].unique())]

    counters: dict[str, Counter] = {"__all__": Counter()}
    for label_id, name in enumerate(label_names):
        tokens = (
            df.loc[df[label_col] == label_id, text_col]
            .str.split()
            .explode()
            .dropna()
        )
        counters[name] = Counter(tokens)
        counters["__all__"].update(counters[name])

    return counters


# ── SOLUTION ──────────────────────────────────────────────────────────────────
train_df["text_clean"] = train_df["text"].apply(preprocess_text)
freq = build_vocab_freq(
    train_df,
    text_col="text_clean",
    label_names=meta["label_names"],
)

print(f"Global vocabulary size : {len(freq['__all__']):,}")
for name in meta["label_names"]:
    print(f"  {name:12s} vocab : {len(freq[name]):,} unique tokens")

#### 6.1.1 — Top-20 tokens per class

In [ ]:
def plot_top_tokens(
    counters: dict[str, Counter],
    top_n: int = 20,
    exclude_labels: list[str] | None = None,
    figsize_per_class: tuple[float, float] = (5.0, 4.5),
) -> None:
    """Horizontal bar chart of the top-n tokens for each class.

    Parameters
    ----------
    counters : dict[str, Counter]
        Output of ``build_vocab_freq``.
    top_n : int
        Number of top tokens to display per class.
    exclude_labels : list[str] or None
        Labels to skip, e.g. ``["__all__"]``.
    figsize_per_class : tuple[float, float]
        Width and height per subplot panel.
    """
    exclude = set(exclude_labels or ["__all__"])
    class_names = [k for k in counters if k not in exclude]
    n = len(class_names)
    fig, axes = plt.subplots(
        1, n,
        figsize=(figsize_per_class[0] * n, figsize_per_class[1]),
    )
    if n == 1:
        axes = [axes]

    palette = sns.color_palette("muted", n)
    for ax, name, colour in zip(axes, class_names, palette):
        tokens, counts = zip(*counters[name].most_common(top_n))
        ax.barh(list(tokens)[::-1], list(counts)[::-1], color=colour)
        ax.set_title(name, fontsize=11, fontweight="bold")
        ax.set_xlabel("count")
        ax.tick_params(axis="y", labelsize=8)

    fig.suptitle(f"Top-{top_n} tokens per class (training set)", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# ── SOLUTION ──────────────────────────────────────────────────────────────────
plot_top_tokens(freq, top_n=20)

#### 6.1.2 — Zipf curve

Plot **log(rank)** vs **log(frequency)** for the full training vocabulary.
A straight line with slope ≈ −1 is the signature of Zipf's law.

In [ ]:
def plot_zipf(
    counter: Counter,
    max_rank: int = 10_000,
    title: str = "Zipf's Law — training vocabulary",
) -> None:
    """Log-log plot of rank vs frequency to visualise Zipf's law.

    Parameters
    ----------
    counter : Counter
        Global token frequency counter (e.g. ``counters["__all__"]``).
    max_rank : int
        Number of tokens (by descending frequency) to plot.
    title : str
        Plot title.
    """
    freqs = sorted(counter.values(), reverse=True)[:max_rank]
    ranks = range(1, len(freqs) + 1)

    plt.figure(figsize=(7, 4))
    plt.loglog(ranks, freqs, lw=1.5, color="steelblue", label="observed")

    # Ideal Zipf reference line: freq ∝ 1/rank
    ref = [freqs[0] / r for r in ranks]
    plt.loglog(ranks, ref, "--", color="salmon", alpha=0.7, label="ideal Zipf (slope = -1)")

    plt.xlabel("Rank (log scale)")
    plt.ylabel("Frequency (log scale)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ── SOLUTION ──────────────────────────────────────────────────────────────────
plot_zipf(freq["__all__"])

#### 6.1.3 — Vocabulary overlap between classes

A token that appears in every class carries no discriminative signal.
Compute the **Jaccard similarity** between each pair of class top-*k* vocabularies.

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

In [ ]:
def vocab_overlap_matrix(
    counters: dict[str, Counter],
    top_k: int = 500,
    exclude_labels: list[str] | None = None,
) -> pd.DataFrame:
    """Pairwise Jaccard similarity between the top-k vocabularies of each class.

    Parameters
    ----------
    counters : dict[str, Counter]
        Output of ``build_vocab_freq``.
    top_k : int
        Number of most-frequent tokens per class to compare.
    exclude_labels : list[str] or None
        Labels to skip.

    Returns
    -------
    pd.DataFrame
        Square matrix of Jaccard similarities, index and columns = class names.
    """
    exclude = set(exclude_labels or ["__all__"])
    names = [k for k in counters if k not in exclude]
    top_sets = {n: set(t for t, _ in counters[n].most_common(top_k)) for n in names}

    matrix = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b in names:
            inter = len(top_sets[a] & top_sets[b])
            union = len(top_sets[a] | top_sets[b])
            matrix.loc[a, b] = inter / union if union else 0.0

    return matrix


# ── SOLUTION ──────────────────────────────────────────────────────────────────
overlap = vocab_overlap_matrix(freq, top_k=500)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    overlap.astype(float),
    annot=True,
    fmt=".2f",
    vmin=0, vmax=1,
    cmap="Blues",
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Pairwise Jaccard similarity — top-500 tokens per class")
plt.tight_layout()
plt.show()

### 📝 Your analysis — Exercise 6.1

**Q1.** What are the top-5 tokens in each class? Are they stop words or content words?

**Q2.** Does the Zipf curve match the ideal slope of −1? What does deviation tell you?

**Q3.** Which pair of classes has the highest vocabulary overlap? What does that imply for a classifier?

*Write your answers in this cell.*

---
### Exercise 6.2 — Tokeniser comparison audit

You already ran the three tokenisers in Part 2. Here you go deeper:
measure **vocabulary size**, **OOV rate** (on the validation set), and
**sequence length distributions** for each tokeniser.

> **Expected insight:** whitespace tokenisation has the largest vocabulary and
> the highest OOV rate. BPE has zero OOV by design. NLTK sits between them.

In [ ]:
def tokeniser_audit(
    train_texts: list[str],
    val_texts: list[str],
    tokenisers: dict,
) -> pd.DataFrame:
    """Compare tokenisers on vocabulary size, OOV rate, and sequence statistics.

    Parameters
    ----------
    train_texts : list[str]
        Preprocessed training documents used to build each vocabulary.
    val_texts : list[str]
        Preprocessed validation documents used to measure OOV rate.
    tokenisers : dict
        Mapping of name → callable(str) -> list[str].

    Returns
    -------
    pd.DataFrame
        One row per tokeniser with columns: vocab_size, oov_rate_val,
        mean_len_train, median_len_train, p95_len_train.
    """
    rows = []
    for name, fn in tokenisers.items():
        train_tokens = [tok for text in train_texts for tok in fn(text)]
        vocab = set(train_tokens)

        val_tokens = [tok for text in val_texts for tok in fn(text)]
        oov_count = sum(1 for t in val_tokens if t not in vocab)
        oov_rate  = oov_count / len(val_tokens) if val_tokens else 0.0

        lengths = np.array([len(fn(text)) for text in train_texts])

        rows.append({
            "tokeniser":        name,
            "vocab_size":       len(vocab),
            "oov_rate_val":     round(oov_rate, 4),
            "mean_len_train":   round(lengths.mean(), 1),
            "median_len_train": int(np.median(lengths)),
            "p95_len_train":    int(np.percentile(lengths, 95)),
        })

    return pd.DataFrame(rows).set_index("tokeniser")


# ── SOLUTION ──────────────────────────────────────────────────────────────────
tokenisers = {
    "whitespace": tokenize_whitespace,
    "nltk":       tokenize_nltk,
    "bpe":        tokenize_bpe,
}

train_texts_clean = train_df["text_clean"].tolist()
val_texts_clean   = val_df["text"].apply(preprocess_text).tolist()

audit = tokeniser_audit(train_texts_clean, val_texts_clean, tokenisers)
display(audit)

#### 6.2.1 — Sequence length distributions

Plot overlapping histograms of sequence lengths for each tokeniser.

In [ ]:
def plot_length_distributions(
    train_texts: list[str],
    tokenisers: dict,
    bins: int = 50,
) -> None:
    """Overlapping histograms of token sequence lengths for each tokeniser.

    Parameters
    ----------
    train_texts : list[str]
        Preprocessed training documents.
    tokenisers : dict
        Mapping of name → callable(str) -> list[str].
    bins : int
        Number of histogram bins.
    """
    fig, ax = plt.subplots(figsize=(8, 4))
    palette = sns.color_palette("muted", len(tokenisers))

    for (name, fn), colour in zip(tokenisers.items(), palette):
        lengths = [len(fn(text)) for text in train_texts]
        ax.hist(lengths, bins=bins, alpha=0.5, label=name, color=colour, density=True)

    ax.set_xlabel("Sequence length (tokens)")
    ax.set_ylabel("Density")
    ax.set_title("Token sequence length distribution by tokeniser")
    ax.legend()
    plt.tight_layout()
    plt.show()


# ── SOLUTION ──────────────────────────────────────────────────────────────────
plot_length_distributions(train_texts_clean, tokenisers)

### 📝 Your analysis — Exercise 6.2

**Q1.** Which tokeniser produces the largest vocabulary? Is a larger vocabulary always better?

**Q2.** What is the OOV rate for whitespace tokenisation on the validation set?
What does that mean for a model trained with this tokeniser?

**Q3.** Look at the p95 sequence lengths. How would these numbers affect a model that
has a maximum context window (e.g., BERT's 512 tokens)?

*Write your answers in this cell.*

---
### Exercise 6.3 — Preprocessing ablation

Each normalisation step changes the vocabulary. Some changes help; some are neutral;
some lose information. Here you measure the effect of each step **empirically**.

You will:
1. Build variants of the pipeline, each skipping one step.
2. Measure vocabulary size for each variant.
3. Compute χ² scores for the top tokens under each variant to measure class separability.

> **Expected insight:** lowercasing has the largest effect on vocabulary size.
> Stop word removal reduces vocabulary moderately but can hurt class separability
> if class-specific function words exist.

In [ ]:
def ablation_vocab_sizes(
    texts: list[str],
    pipelines: dict[str, callable],
) -> pd.DataFrame:
    """Measure vocabulary size for each preprocessing pipeline variant.

    Parameters
    ----------
    texts : list[str]
        Raw (unnormalised) documents.
    pipelines : dict[str, callable]
        Mapping of pipeline name → function(str) -> str.

    Returns
    -------
    pd.DataFrame
        One row per pipeline: pipeline name and vocabulary size.
    """
    rows = []
    for name, fn in pipelines.items():
        vocab = set(
            tok
            for text in texts
            for tok in fn(text).split()
        )
        rows.append({"pipeline": name, "vocab_size": len(vocab)})

    return pd.DataFrame(rows).set_index("pipeline").sort_values(
        "vocab_size", ascending=False
    )


# ── SOLUTION — pipeline variants ─────────────────────────────────────────────
def preprocess_no_html(text: str) -> str:
    """Full pipeline minus the HTML stripping step.

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string without HTML removal.
    """
    text = text.lower()
    # HTML step skipped
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r" +", " ", text)
    return text.strip()


def preprocess_no_punct(text: str) -> str:
    """Full pipeline minus the punctuation removal step.

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string retaining punctuation.
    """
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    # Punctuation removal step skipped
    text = re.sub(r" +", " ", text)
    return text.strip()


pipelines = {
    "full pipeline":    preprocess_text,
    "no lowercase":     preprocess_no_lowercase,
    "no html strip":    preprocess_no_html,
    "no punct removal": preprocess_no_punct,
}

raw_texts = train_df["text"].tolist()
sizes = ablation_vocab_sizes(raw_texts, pipelines)
display(sizes)

#### 6.3.1 — Class separability: χ² of top tokens

For each pipeline variant, compute the **χ² score** of the top-50 tokens
against the class labels. A higher mean χ² means the token distribution
is more informative for classification.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2


def mean_chi2_top_tokens(
    train_texts: list[str],
    train_labels: list[int],
    pipeline_fn: callable,
    top_n: int = 50,
) -> float:
    """Mean χ² score of the top-n tokens for a given preprocessing pipeline.

    Fits a unigram CountVectorizer on preprocessed training text, computes
    the χ² statistic between each token and the class labels, and returns
    the mean over the top-n tokens by χ² score.

    Parameters
    ----------
    train_texts : list[str]
        Raw (unnormalised) training documents.
    train_labels : list[int]
        Integer class labels for each training document.
    pipeline_fn : callable
        Preprocessing function str -> str.
    top_n : int
        Number of top tokens (by χ² score) to average over.

    Returns
    -------
    float
        Mean χ² score of the top-n tokens.
    """
    cleaned = [pipeline_fn(t) for t in train_texts]
    vec = CountVectorizer(max_features=5_000)
    X = vec.fit_transform(cleaned)
    scores, _ = chi2(X, train_labels)
    return float(np.sort(scores)[::-1][:top_n].mean())


# ── SOLUTION ──────────────────────────────────────────────────────────────────
train_labels = train_df["label"].tolist()

ablation_summary = []
for name, fn in pipelines.items():
    vocab_size = sizes.loc[name, "vocab_size"] if name in sizes.index else None
    chi2_score = mean_chi2_top_tokens(raw_texts, train_labels, fn)
    ablation_summary.append({
        "pipeline":  name,
        "vocab_size": vocab_size,
        "mean_chi2":  round(chi2_score, 1),
    })

ablation_df = pd.DataFrame(ablation_summary).set_index("pipeline")
display(ablation_df)

# Scatter: vocab_size vs mean_chi2
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(
    ablation_df["vocab_size"],
    ablation_df["mean_chi2"],
    s=80, zorder=5,
)
for row in ablation_df.itertuples():
    ax.annotate(
        row.Index,
        (row.vocab_size, row.mean_chi2),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=9,
    )
ax.set_xlabel("Vocabulary size")
ax.set_ylabel("Mean χ² (top-50 tokens)")
ax.set_title("Preprocessing ablation: vocab size vs class separability")
plt.tight_layout()
plt.show()

### 📝 Your analysis — Exercise 6.3

**Q1.** Which preprocessing step reduces vocabulary size the most?

**Q2.** Does a smaller vocabulary always lead to higher χ² (better separability)?
Find a case where removing a step hurts separability.

**Q3.** Based on your results, which pipeline variant would you carry into Session 3?

*Write your answers in this cell.*

---
### Exercise 6.4 — Class fingerprinting with TF-IDF ⭐ *Homework*

> Complete this exercise before **Session 2**. It will be reviewed at the start of class.

Raw frequency tells you which tokens are common. TF-IDF tells you which tokens are
**distinctive** — frequent in one class but rare across the others.

You will:
1. Compute a TF-IDF matrix over the training set.
2. For each class, average TF-IDF scores across all documents in that class.
3. Extract the **top-15 discriminative tokens per class**.
4. Visualise the result as a **heatmap** (classes × top tokens).

This is the closest you can get to "understanding your data" without training a model.
If the heatmap shows clear per-class clusters, a classifier will likely do well.
If everything is mixed, you have a hard problem.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np


def compute_class_tfidf(
    train_df: pd.DataFrame,
    text_col: str = "text_clean",
    label_col: str = "label",
    label_names: list[str] | None = None,
    max_features: int = 10_000,
    top_n: int = 15,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compute mean per-class TF-IDF scores and extract top discriminative tokens.

    Fits a TF-IDF vectoriser on the training set, then for each class averages
    the TF-IDF scores of all documents belonging to that class.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training set with preprocessed text and integer labels.
    text_col : str
        Column containing preprocessed text.
    label_col : str
        Column containing integer class labels.
    label_names : list[str] or None
        Human-readable label names. If None, uses str(label).
    max_features : int
        Maximum vocabulary size for the TF-IDF vectoriser.
    top_n : int
        Number of top tokens to extract per class.

    Returns
    -------
    mean_tfidf : pd.DataFrame
        Shape (n_classes, max_features). Mean TF-IDF score per class per token.
    top_tokens : pd.DataFrame
        Shape (n_classes, top_n). Top discriminative tokens per class.
    """
    if label_names is None:
        label_names = [str(l) for l in sorted(train_df[label_col].unique())]

    vec = TfidfVectorizer(max_features=max_features)
    X = vec.fit_transform(train_df[text_col])
    feature_names = vec.get_feature_names_out()

    # For each class: compute mean TF-IDF across all documents in that class
    rows = {}
    for label_id, name in enumerate(label_names):
        mask = (train_df[label_col] == label_id).values
        # Use sparse slicing + mean to avoid materialising a dense matrix
        class_mean = np.asarray(X[mask].mean(axis=0)).flatten()
        rows[name] = class_mean

    mean_tfidf = pd.DataFrame(rows, index=feature_names).T
    # mean_tfidf shape: (n_classes, max_features)

    # Top-n tokens per class by mean TF-IDF score
    top_tokens = pd.DataFrame(
        {
            name: mean_tfidf.loc[name].nlargest(top_n).index.tolist()
            for name in label_names
        },
        index=range(1, top_n + 1),
    )

    return mean_tfidf, top_tokens


def plot_class_fingerprint(
    mean_tfidf: pd.DataFrame,
    top_n: int = 15,
    figsize: tuple[float, float] = (14, 5),
) -> None:
    """Heatmap of the top discriminative tokens per class.

    For each class, selects the top-n tokens by mean TF-IDF score,
    takes the union of all selected tokens, and plots a heatmap showing
    the mean TF-IDF score of each token across all classes.

    Parameters
    ----------
    mean_tfidf : pd.DataFrame
        Output of ``compute_class_tfidf`` — mean TF-IDF per class per token.
    top_n : int
        Number of top tokens to select per class before taking the union.
    figsize : tuple[float, float]
        Figure size.
    """
    # Collect union of top-n tokens across all classes
    selected_tokens: set[str] = set()
    for class_name in mean_tfidf.index:
        top = mean_tfidf.loc[class_name].nlargest(top_n).index.tolist()
        selected_tokens.update(top)

    # Slice the mean_tfidf matrix to the selected tokens only
    plot_data = mean_tfidf[sorted(selected_tokens)]

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        plot_data,
        cmap="YlOrRd",
        annot=False,
        linewidths=0.3,
        ax=ax,
        xticklabels=True,
    )
    ax.set_title(f"Class fingerprint — top-{top_n} TF-IDF tokens per class (union)")
    ax.set_xlabel("Token")
    ax.set_ylabel("Class")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    plt.tight_layout()
    plt.show()


# ── SOLUTION ──────────────────────────────────────────────────────────────────
mean_tfidf, top_tokens = compute_class_tfidf(
    train_df,
    text_col="text_clean",
    label_names=meta["label_names"],
)

print("Top-15 discriminative tokens per class:")
display(top_tokens)

plot_class_fingerprint(mean_tfidf)

### 📝 Your analysis — Exercise 6.4 (Homework)

**Q1.** For each class, list the top-5 tokens from the heatmap.
Do they match your intuition about what that class is about?

**Q2.** Are there any tokens that score highly across **multiple** classes?
What does that tell you about how easy or hard this classification problem is?

**Q3.** Pick one class pair (e.g., Sports vs Business for AG News).
List 3 tokens that strongly separate them. Would a simple frequency-based rule
using those tokens work as a classifier?

**Q4.** (Optional challenge) Modify `compute_class_tfidf` to use
**sublinear TF scaling** (`sublinear_tf=True` in `TfidfVectorizer`).
Does the heatmap change? Why?

*Write your answers in this cell.*

---
## Summary

Run the cell below to print your final results table before the end-of-session review.

In [ ]:
print(f"Dataset    : {DATASET_NAME}")
print(f"Train size : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}")
print(f"Classes    : {meta['num_classes']}  ({', '.join(meta['label_names'])})")
print()

# Majority baseline metrics
from sklearn.metrics import accuracy_score, f1_score
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Weighted F1"],
    "Majority Baseline (test)": [
        f"{accuracy_score(y_true_test, y_pred_test):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='macro',    zero_division=0):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0):.3f}",
    ],
})
display(summary)
print()
print("Tokeniser comparison:")
display(comparison)

---
## What's next?

In **TP 02** you will train n-gram language models on this dataset and measure perplexity.
In **TP 03** you will replace the majority baseline with Naive Bayes and Logistic Regression —
the class fingerprinting from Exercise 6.4 will tell you exactly which tokens those models will rely on.

**Keep this notebook open.** The objects `train_df`, `val_df`, `test_df`, `meta`, and `comparison`
carry forward into every future session.

---
## 🗒 Instructor Notes — Part 6

### Expected results on AG News (train split)

**6.1 — Top tokens**
The top-5 tokens in every class will be stop words (`the`, `of`, `in`, `a`, `to`).
This is the Zipf effect in action — use it to motivate TF-IDF before Session 3.
The Zipf curve should track the ideal slope closely down to rank ~2,000, then flatten
(words with count 1–2 deviate from the law).

Vocabulary overlap (Jaccard, top-500):
- World / Sci/Tech: highest overlap (~0.55) — both contain technical and geopolitical terms.
- Sports / Business: lowest overlap (~0.28) — good class separation expected.

**6.2 — Tokeniser audit**

| tokeniser | vocab_size | oov_rate_val | p95_len |
|-----------|-----------|--------------|---------|
| whitespace | ~45,000 | ~2–4% | ~55 |
| nltk | ~48,000 | ~2–4% | ~60 |
| bpe | ~5,000 | 0% | ~80 |

BPE OOV = 0 by construction. Whitespace OOV comes from proper nouns and rare compounds seen only in val.

**6.3 — Ablation**
`no punct removal` will have the largest vocabulary (~2–3× the full pipeline).
`no lowercase` will be ~1.3–1.5× larger.
χ² scores should be highest for the full pipeline — every step helps separability on this corpus.

**6.4 — Class fingerprinting (homework)**
Expected top tokens per AG News class:
- World: `government`, `president`, `military`, `war`, `minister`
- Sports: `game`, `season`, `cup`, `match`, `championship`
- Business: `company`, `percent`, `market`, `billion`, `shares`
- Sci/Tech: `software`, `microsoft`, `technology`, `internet`, `windows`

The heatmap should show clean per-class clusters with minimal cross-class leakage.
If students see `said` or `year` as top tokens for a class, they forgot stop word removal
or the pipeline was not applied — use as a debugging teachable moment.

### Common mistakes
- Applying preprocessing **after** building the frequency counter (vocab is raw text).
- Forgetting to apply `preprocess_text` to val/test before the tokeniser audit OOV computation.
- In 6.4, materialising the full dense TF-IDF matrix for large corpora — the sparse mean avoids this.
- Confusing `top_tokens` (a table of token names) with `mean_tfidf` (a matrix of scores).